# ICLR-II Flat Notebook
This notebook is a flat, self-contained version of `iclrii.py`. All repo-level dependencies are defined below (no local imports). Adjust hyperparameters at the top, then run top-to-bottom to reproduce the three stages: (A,B) ensemble, x0 scores, and estimator errors.

In [ ]:
# Imports (Python libraries only)
import argparse
import math
import pathlib
import warnings
from dataclasses import dataclass
from typing import Any, Dict, Iterable, Mapping, Optional, Sequence, Tuple

import numpy as np
import numpy.linalg as npl
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib import colormaps

import scipy.linalg
from scipy.linalg import expm, eigvals, solve_continuous_lyapunov, solve_discrete_lyapunov, null_space, qr

import pysindy
import torch

from IPython.display import display

## Hyperparameters
Set these to match the CLI flags in `iclrii.py`.

In [ ]:
# Output directories
base_outdir = "pyident_results"
ensemble_dir = "fresh_ensemble"
x0_dir = "fresh_ABx0"
pbh_dir = "fresh_nonidentifiable_ABx0"

# Ensemble sweep
sparsity_grid = "0.0:0.1:1.0"
ndim_grid = "2:1:10"
samples = 10000

# x0 sampling and plotting
x0_samples = 10
outlier_trim = 0.05
sphere = True
sparse_p = None  # set a float in (0,1] to use masked x0

# Estimation / simulation
seed = 12345
T = 100
dt = 1.0
u_scale = 3.0
dwell = 1
algos = "DMDc"  # e.g., "SINDy,DMDc,MOESP,NODE"

input_family = "prbs"  # "prbs" or "multisine"
pe_method = "block"    # "block" or "moment"
pe_tol = 1e-8
pe_max_tries = 128

dmdc_z_cond_max = 1e8

# x0 constraints
x0_min_norm = 1e-6
x0_min_support = 1
x0_max_attempts = 128
min_visible_dim = 0

# Selection/output tweaks
selected_suffix = None  # override selected dataset suffix (default: all_x0)

# DMDc ridge option
ridge = False
ridge_lam = 1e-6

## Dependencies
Below are the repo-level dependencies, inlined and ordered by use.

In [ ]:
# Local dependency imports (from existing repo files)
import sys
import pathlib

_repo_root = pathlib.Path.cwd()
if (_repo_root / "__init__.py").exists() and _repo_root.name == "pyident":
    sys.path.insert(0, str(_repo_root.parent))

import pyident.loggers.tolerances as tolerances
import pyident.signals as signals
import pyident.pe_sig as pe_sig
import pyident.simulation as simulation
import pyident.experiments.unctrb_utils as unctrb_utils
import pyident.ensembles as ensembles
import pyident.metrics as metrics
import pyident.projectors as projectors
import pyident.experiments.boxplot_style as boxplot_style
import pyident.estimators as estimators

## Experiment Modules (Flattened)
These are the experiment scripts used by `iclrii.py`, now inlined.

In [ ]:
# Experiment modules
import pyident.experiments.sim_regcomb as sim_regcomb
import pyident.experiments.sim_regcomb_ctrb as sim_regcomb_ctrb
import pyident.experiments.sim_unctrb_x0_boxplot as sim_unctrb_x0_boxplot
import pyident.experiments.sim_unctrb_pbh_estimators as sim_unctrb_pbh_estimators
import pyident.experiments.sim_unctrb_pbh_error_boxplots as sim_unctrb_pbh_error_boxplots

# Aliases to match the flat notebook interface
sim_regcomb_ctrb_run = sim_regcomb_ctrb.run
sim_regcomb_ctrb_build_parser = sim_regcomb_ctrb.build_parser

sim_unctrb_x0_boxplot_run = sim_unctrb_x0_boxplot.run
sim_unctrb_x0_boxplot_build_parser = sim_unctrb_x0_boxplot.build_parser

sim_unctrb_pbh_estimators_run = sim_unctrb_pbh_estimators.run
sim_unctrb_pbh_estimators_build_parser = sim_unctrb_pbh_estimators.build_parser

sim_unctrb_pbh_error_boxplots_run = sim_unctrb_pbh_error_boxplots.run
sim_unctrb_pbh_error_boxplots_build_parser = sim_unctrb_pbh_error_boxplots.build_parser

## Pipeline
Runs the three stages (A,B), x0 scores, and estimator errors. Outputs are saved to disk and previewed below.

In [ ]:
from types import SimpleNamespace


def _mask_ps_from_scheme(*, sphere: bool, sparse_p: Optional[float]) -> list[float]:
    if sparse_p is not None and sphere:
        raise ValueError("Use either sphere=True or sparse_p, not both.")
    if sparse_p is not None:
        p = float(sparse_p)
        if not (0.0 < p <= 1.0):
            raise ValueError("sparse_p must be in (0, 1].")
        return [p]
    return [1.0]  # default sphere


def run_iclrii_pipeline():
    base_out = pathlib.Path(base_outdir)
    ensemble_out = base_out / ensemble_dir
    x0_out = base_out / x0_dir
    pbh_out = base_out / pbh_dir
    boxplot_out = pbh_out / "boxplots"

    # Stage 1: (A,B) ensemble
    print("Stage 1: Generating (A,B) ensemble...")
    regcomb_args = sim_regcomb_ctrb_build_parser().parse_args(
        [
            "--axes",
            "sparsity,ndim",
            "--sparsity-grid",
            sparsity_grid,
            "--ndim-grid",
            ndim_grid,
            "--samples",
            str(samples),
            "--outdir",
            str(ensemble_out),
            "--save-matrices",
        ]
    )
    if regcomb_args.m is None:
        regcomb_args.m = int(regcomb_args.n)
    sim_regcomb_ctrb_run(regcomb_args)

    systems_csv = ensemble_out / "systems.csv"
    systems_npz = ensemble_out / "systems_matrices.npz"
    summary_csv = ensemble_out / "scores_summary.csv"

    systems_df = pd.read_csv(systems_csv)
    summary_df = pd.read_csv(summary_csv)
    print(f"Saved {len(systems_df)} systems -> {systems_csv}")
    display(systems_df.head())
    display(summary_df.head())

    # Stage 2: x0 identifiability scores
    mask_ps = _mask_ps_from_scheme(sphere=sphere, sparse_p=sparse_p)
    print("Stage 2: Computing x0 identifiability scores...")
    x0_args = sim_unctrb_x0_boxplot_build_parser().parse_args(
        [
            "--dataset-csv",
            str(systems_csv),
            "--dataset-npz",
            str(systems_npz),
            "--x0-samples",
            str(x0_samples),
            "--mask-ps",
            *[str(p) for p in mask_ps],
            "--mask-renorm",
            "--x0-min-norm",
            str(x0_min_norm),
            "--x0-min-support",
            str(x0_min_support),
            "--x0-max-attempts",
            str(x0_max_attempts),
            "--outdir",
            str(x0_out),
            "--outlier-trim",
            str(outlier_trim),
        ]
    )
    sim_unctrb_x0_boxplot_run(x0_args)

    scores_csv = x0_out / "identifiability_scores.csv"
    scores_df = pd.read_csv(scores_csv)
    print(f"Saved {len(scores_df)} score rows -> {scores_csv}")
    display(scores_df.head())

    # Stage 3: Estimator errors (NODE, DMDc, MOESP, SINDy)
    print("Stage 3: Running estimators...")
    pbh_threshold = float("inf")
    suffix = selected_suffix or "all_x0"

    pbh_args = sim_unctrb_pbh_estimators_build_parser().parse_args(
        [
            "--dataset-csv",
            str(systems_csv),
            "--dataset-npz",
            str(systems_npz),
            "--outdir",
            str(pbh_out),
            "--seed",
            str(seed),
            "--x0-samples",
            str(x0_samples),
            "--mask-ps",
            *[str(p) for p in mask_ps],
            "--mask-renorm",
            "--x0-min-norm",
            str(x0_min_norm),
            "--x0-min-support",
            str(x0_min_support),
            "--x0-max-attempts",
            str(x0_max_attempts),
            "--pbh-threshold",
            str(pbh_threshold),
            "--suffix",
            suffix,
            "--min-visible-dim",
            str(min_visible_dim),
            "--T",
            str(T),
            "--dt",
            str(dt),
            "--u-scale",
            str(u_scale),
            "--dwell",
            str(dwell),
            "--input-family",
            input_family,
            "--pe-method",
            pe_method,
            "--pe-tol",
            str(pe_tol),
            "--pe-max-tries",
            str(pe_max_tries),
            "--algos",
            algos,
            "--dmdc-z-cond-max",
            str(dmdc_z_cond_max),
        ]
        + (["--ridge", "--ridge-lam", str(ridge_lam)] if ridge else [])
    )
    sim_unctrb_pbh_estimators_run(pbh_args)

    selected_csv = pbh_out / f"selected_{suffix}.csv"
    selected_npz = pbh_out / f"selected_{suffix}.npz"
    errors_csv = pbh_out / "estimation_errors.csv"

    selected_df = pd.read_csv(selected_csv)
    errors_df = pd.read_csv(errors_csv)
    print(f"Saved {len(selected_df)} selected triples -> {selected_csv}")
    display(selected_df.head())
    display(errors_df.head())

    # Stage 4: Error boxplots
    print("Stage 4: Error boxplots...")
    box_args = sim_unctrb_pbh_error_boxplots_build_parser().parse_args(
        [
            "--selected-npz",
            str(selected_npz),
            "--selected-csv",
            str(selected_csv),
            "--outdir",
            str(boxplot_out),
            "--seed",
            str(seed),
            "--min-visible-dim",
            str(min_visible_dim),
            "--T",
            str(T),
            "--dt",
            str(dt),
            "--u-scale",
            str(u_scale),
            "--dwell",
            str(dwell),
            "--input-family",
            input_family,
            "--pe-method",
            pe_method,
            "--pe-tol",
            str(pe_tol),
            "--pe-max-tries",
            str(pe_max_tries),
            "--algos",
            algos,
            "--dmdc-z-cond-max",
            str(dmdc_z_cond_max),
        ]
        + (["--ridge", "--ridge-lam", str(ridge_lam)] if ridge else [])
    )
    sim_unctrb_pbh_error_boxplots_run(box_args)

    box_errors_csv = boxplot_out / "estimation_errors_boxplot.csv"
    box_errors_df = pd.read_csv(box_errors_csv)
    print(f"Saved {len(box_errors_df)} boxplot rows -> {box_errors_csv}")
    display(box_errors_df.head())

    return {
        "systems": systems_df,
        "scores": scores_df,
        "selected": selected_df,
        "errors": errors_df,
        "boxplot_errors": box_errors_df,
        "paths": {
            "ensemble_out": ensemble_out,
            "x0_out": x0_out,
            "pbh_out": pbh_out,
            "boxplot_out": boxplot_out,
        },
    }


# Run the pipeline top-to-bottom
results = run_iclrii_pipeline()